In [1]:
!pip install -q langchain langchain-openai gradio

import gradio as gr
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
import os

def generate_email(company_name, product_name, target_audience, tone, key_benefits, api_key):
    if not api_key:
        return "Error: OpenAI API Key is missing."

    os.environ["OPENAI_API_KEY"] = api_key

    try:
        llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7)

        prompt = PromptTemplate(
            input_variables=["company", "product", "audience", "tone", "benefits"],
            template="""Write a highly converting, personalized cold email.

            Sender Company: {company}
            Product/Service: {product}
            Target Audience: {audience}
            Key Benefits: {benefits}
            Tone: {tone}

            Include a catchy subject line. Keep the body under 150 words. Include a clear call to action (CTA)."""
        )

        chain = prompt | llm
        response = chain.invoke({
            "company": company_name,
            "product": product_name,
            "audience": target_audience,
            "tone": tone,
            "benefits": key_benefits
        })

        return response.content

    except Exception as e:
        return f"Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# ✉️ Automated Cold Email Generator")

    with gr.Row():
        api_key_input = gr.Textbox(label="OpenAI API Key", type="password")

    with gr.Row():
        with gr.Column():
            company_input = gr.Textbox(label="Your Company Name")
            product_input = gr.Textbox(label="Product or Service")
            audience_input = gr.Textbox(label="Target Audience (e.g., HR Directors)")
            benefits_input = gr.Textbox(label="Key Benefits", lines=2)
            tone_input = gr.Dropdown(choices=["Professional", "Casual", "Urgent", "Friendly"], value="Professional", label="Email Tone")
            generate_btn = gr.Button("Generate Email", variant="primary")

        with gr.Column():
            email_output = gr.Textbox(label="Generated Cold Email", lines=15, interactive=False)

    generate_btn.click(
        fn=generate_email,
        inputs=[company_input, product_input, audience_input, tone_input, benefits_input, api_key_input],
        outputs=email_output
    )

demo.launch(share=True)